# TRINETRA · Account F · FUSION (final, fixed)
**A (fire), B (zero-shot) aur preferably C (deepfake) ke same-phase (final) runs khatam hone ke baad chalao.**

**Kaise chalayein:** Import Notebook → **Add Input → Notebook Output** → apne finished A, B, C (aur D, P agar master chahiye) runs →
(team dataset `happyparsi/trinetra-dataset` optional hai) → **Internet ON** (GPU optional, CPU kaafi hai) → **Save & Run All (commit)**.

**Purane F_final se kya theek hua:**
- Data cell ab **61 GB dataset ko walk nahi karta**: F koi image convert nahi karta, sirf A/B/C ke outputs padhta hai.
- Master cell ab `/kaggle/input/**` par recursive glob nahi karta (wo poore 61 GB me zip dhoondhta tha, ghanton atak sakta tha);
  sirf attached run outputs me `out_*.zip` dhoondhta hai.
- Live log + `[heartbeat]` + har command ka `[time]`; offline RL ab `[rl] seed … epoch …/… loss … ETA` print karta hai.
- Timeouts (fuse 120 / har geo-fusion 20 / merge 30 min) taaki koi step chupchaap 12 h tak na chale.

Output: `out_F_final.zip` aur **`master_final.zip`** (`master/MASTER_RESULTS.md`) — ye AI ko bhejo. Numbers = OUR RESULTS on a prototype subset.

**Import:** pehle `/kaggle/working/out_<ACC>_final.zip` (agar A isi session me chala tha), phir attached run outputs (`/kaggle/input`, kisi bhi layout me), phir `EXTRA_RUN_DIRS`. Session ka purana `out/` backup folder me shift hota hai taaki phase2 aur final mix na hon. **B (zero-shot) na mile** aur session me `npz/fire.npz` + GPU ho to B **yahin chal jaata hai** (`AUTO_RUN_B`, ~10–20 min). Phir bhi na ho to notebook fusion se pehle ruk kar batata hai kya karna hai (`REQUIRE_B = False` = sirf fire + colour baseline).

In [ ]:
ACCOUNT = "F"        # A = fire CNN + colour baseline + video | B = zero-shot | C = deepfake | D = EO heads + geo | F = fusion + RL (after A-C) | P = 7 physics models (tests + real-data hindcast, GLOF, RL env, Geo-PINN) | ALL = everything in one session
PHASE   = "final"   # "phase2" = fast (1 seed, few epochs) | "final" = 3 seeds, full epochs
DATASET = "/kaggle/input/trinetra-dataset"
READY   = "/kaggle/input/trinetra-ready-v1/npz"   # used if a teammate already published converted NPZs
GITHUB  = ""         # optional fallback: https URL of a repo that contains TRINETRA/
import os
os.environ["PYTHONUNBUFFERED"] = "1"     # child Python processes print immediately
TIMEOUT_MIN = {"fuse": 120, "geo": 20, "merge": 30, "zeroshot": 60}
REQUIRE_B = True          # True: B (zero-shot) attach na ho to fusion se PEHLE saaf message ke saath ruko
EXTRA_RUN_DIRS = []       # optional: run output folders haath se, e.g. ["/kaggle/input/notebooks/me/trinetra-b-phase2"]
AUTO_RUN_B = True         # B ka output na mile aur isi session me npz/fire.npz + GPU ho to zero-shot YAHIN chala do

In [ ]:
import base64, io, zipfile, os
_Z = '...'
zipfile.ZipFile(io.BytesIO(base64.b64decode(_Z))).extractall('/kaggle/working')
print('embedded TRINETRA code unpacked:', os.path.isdir('/kaggle/working/TRINETRA/trinetra/kaggle'))

In [ ]:
import subprocess, shutil, glob, pathlib, time, threading, signal
def sh(cmd, timeout_min=None, tag=None):
    """Run a shell command with live output, a heartbeat every ~5 silent minutes and an optional hard timeout."""
    tag = tag or cmd.split()[0]; print("$", cmd, flush=True); t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, start_new_session=True)
    last = [time.time()]
    def beat():
        while p.poll() is None:
            time.sleep(30)
            if p.poll() is None and time.time() - last[0] > 290:
                print(f"[heartbeat] {tag} still running · {(time.time() - t0) / 60:.0f} min", flush=True); last[0] = time.time()
    threading.Thread(target=beat, daemon=True).start()
    def kill():
        print(f"[timeout] {tag} ran > {timeout_min} min → stopping it (partial outputs are kept)", flush=True)
        try: os.killpg(p.pid, signal.SIGKILL)
        except ProcessLookupError: pass
    timer = threading.Timer(timeout_min * 60, kill) if timeout_min else None
    if timer: timer.start()
    for line in p.stdout:
        print(line, end="", flush=True); last[0] = time.time()
    rc = p.wait()
    if timer: timer.cancel()
    print(f"[time] {tag}: {(time.time() - t0) / 60:.1f} min · exit {rc}", flush=True)
    return rc
# 1) locate code
hits = glob.glob("/kaggle/input/**/trinetra/kaggle/run_hackathon.sh", recursive=True)
CODE = "/kaggle/working/TRINETRA"
if os.path.isdir(CODE + "/trinetra/kaggle"):
    pass  # embedded package already unpacked
elif hits:
    shutil.copytree(str(pathlib.Path(hits[0]).parents[2]), CODE, dirs_exist_ok=True)
elif GITHUB:
    sh(f"git clone --depth 1 {GITHUB} /kaggle/working/repo && cp -r /kaggle/working/repo/TRINETRA {CODE}")
else:
    raise SystemExit("Attach the TRINETRA code dataset (or set GITHUB)")
print("code at", CODE); T_RUN = time.time()

In [ ]:
EPOCHS, SEEDS = ("5", "0") if PHASE == "phase2" else ("15", "0 1 2")
os.environ.update(OUT="/kaggle/working/out", EPOCHS=EPOCHS, SEEDS=SEEDS, PY="python",
                  KAGGLE_INPUT=DATASET, DFIRE_N="3000" if PHASE == "phase2" else "4000",
                  CIFAKE_N="5000" if PHASE == "phase2" else "10000")
sh("pip -q install timm tifffile imagecodecs h5py" + (" open_clip_torch" if ACCOUNT in ("B", "ALL") else ""))
sh("nvidia-smi --query-gpu=name,memory.total --format=csv || true")

In [ ]:
# 2) FUSION needs NO image data. Only locate the team dataset (if attached) so it is EXCLUDED from the run-output search.
#    No walk over the 61 GB dataset (that is what made other notebooks look stuck for hours).
import json, re, sys
MARKERS = {"cifake", "ai generated images vs real images", "avenue_dataset", "crisismmd_v2.0"}
def _kids(d):
    try: return {e.lower() for e in os.listdir(d)}
    except OSError: return set()
def find_dataset_root():
    cands = [DATASET] + sorted(glob.glob("/kaggle/input/*")) + sorted(glob.glob("/kaggle/input/*/*")) \
          + sorted(glob.glob("/kaggle/input/*/*/*")) + sorted(glob.glob("/kaggle/input/*/*/*/*"))
    seen = []
    for cand in cands:
        r = os.path.realpath(cand)
        if os.path.isdir(r) and r not in seen:
            seen.append(r)
            if _kids(r) & MARKERS or ("trinetra-dataset" in r.lower() and _kids(r)):
                return r
    return None
DATASET = find_dataset_root()
print("team dataset:", DATASET or "not attached (fine for fusion)")
STAGES = ["fuse"]; NEEDS = {}
os.environ["DATA"] = "/kaggle/working/npz"; os.makedirs(os.environ["DATA"], exist_ok=True)
sh("ls /kaggle/input; nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null || echo 'no GPU (CPU is fine for F)'", tag="inputs")

In [ ]:
# 2b) FUSION ONLY: collect the A / B / C (D, P) results for this PHASE.
#     Sources (all optional, in this order):
#       (1) out_<ACC>_<PHASE>.zip already in /kaggle/working   ← e.g. A ran in THIS same session (your case)
#       (2) attached run outputs anywhere in /kaggle/input      ← Add Input → Your Work / Notebooks → the run → Add
#       (3) EXTRA_RUN_DIRS from the settings cell
#     If B (zero-shot) is still missing and this session has npz/fire.npz + a GPU, B is run HERE (AUTO_RUN_B).
import zipfile, time
WK = "/kaggle/working"; OUTD = f"{WK}/out"; IMP = f"{WK}/imported"
DS_REAL = os.path.realpath(DATASET) if DATASET else None
ART = re.compile(r".*_s\d+_(oof\.npz|report\.json)$")   # seed-tagged stage artifacts, e.g. fire_s0_oof.npz
ZIPRE = re.compile(r"out_([A-Za-z]+)_(phase2|final)\.zip$")

def _is_team(p):
    rp = os.path.realpath(p)
    return bool(DS_REAL) and (rp == DS_REAL or rp.startswith(DS_REAL + "/"))

def find_run_roots(base="/kaggle/input", max_depth=6):
    """BFS over folder listings only; never enters the team dataset or image-heavy folders."""
    roots, q = [], [(base, 0)]
    while q:
        d, dep = q.pop(0)
        if _is_team(d) or (_kids(d) & MARKERS):
            continue
        try:
            with os.scandir(d) as it: ents = list(it)
        except OSError:
            continue
        names = [e.name for e in ents]
        if any(ZIPRE.match(n) for n in names) or any(ART.match(n) for n in names) or "stage_status.json" in names:
            roots.append(d); continue
        if dep < max_depth and len(names) < 500:
            q += [(e.path, dep + 1) for e in ents if e.is_dir()]
    return roots

# --- move this session's old out/ aside so phase2 and final results never mix (nothing is deleted)
BACKUP = None
if os.path.isdir(OUTD) and any(ART.match(f) for _, _, fs in os.walk(OUTD) for f in fs):
    BACKUP = f"{WK}/out_session_backup_{time.strftime('%H%M%S')}"
    shutil.move(OUTD, BACKUP)
    print(f"[import] old {OUTD} (earlier run in this session) moved to {BACKUP} so phases do not mix")
os.makedirs(OUTD, exist_ok=True)

# --- (1) + (2) + (3): candidate zips, then pick ONE per account (this PHASE first, else the other phase)
work_zips = [z for z in glob.glob(f"{WK}/out_*.zip") if ZIPRE.match(os.path.basename(z))
             and ZIPRE.match(os.path.basename(z)).group(1).upper() != "F"]          # never re-import our own output
input_roots = find_run_roots() + [d for d in globals().get("EXTRA_RUN_DIRS", []) if os.path.isdir(d)]
input_roots = [p for p in input_roots if not any(p != q and p.startswith(q + "/") for q in input_roots)]
run_roots = input_roots                                  # also used by the master cell below
print("[import] zips in /kaggle/working:", [os.path.basename(z) for z in work_zips] or "none")
print("[import] attached run outputs:", input_roots or "none")
in_zips = []
for r in input_roots:
    for cur, dirs, files in os.walk(r, followlinks=True):
        in_zips += [os.path.join(cur, f) for f in files if ZIPRE.match(f)]
by_acc = {}
for z in work_zips + sorted(set(in_zips)):                # /kaggle/working first
    acc, ph = ZIPRE.match(os.path.basename(z)).groups()
    by_acc.setdefault(acc.upper(), {}).setdefault(ph, z)
chosen = {}
for acc, d in sorted(by_acc.items()):
    if PHASE in d:
        chosen[acc] = d[PHASE]
    else:
        chosen[acc] = next(iter(d.values()))
        print(f"[import][note] account {acc}: no {PHASE} zip, using {os.path.basename(chosen[acc])} (other phase)")
print("[import] using:", {a: os.path.basename(z) for a, z in chosen.items()} or "no zips")

def _copy_stages(root, skip=()):
    got = []
    for cur, dirs, files in os.walk(root, followlinks=True):
        if "results" in cur.split(os.sep): continue
        name = os.path.basename(cur)
        if name in skip: continue
        if any(ART.match(f) for f in files):
            shutil.copytree(cur, f"{OUTD}/{name}", dirs_exist_ok=True); got.append(name)
            print(f"[import] {name:12s} <- {cur}")
    return got

imported = []
for acc, z in chosen.items():
    dst = f"{IMP}/{pathlib.Path(z).stem}"
    if not os.path.isdir(dst):
        print("unzip", z); zipfile.ZipFile(z).extractall(dst)
    imported += _copy_stages(dst)                          # only the chosen snapshot, never other phases
# raw folders (attached runs without a zip, then this session's backup) fill only what the zips did not provide
zip_roots = {r for r in input_roots if any(os.path.realpath(z).startswith(os.path.realpath(r) + "/") for z in in_zips)}
for r in [r for r in input_roots if r not in zip_roots] + ([BACKUP] if BACKUP and not chosen else []):
    imported += _copy_stages(r, skip=set(imported))

def _have():
    return {k: os.path.exists(f"{OUTD}/{p}") for k, p in
            (("fire (A)", "fire/fire_s0_oof.npz"), ("colorhist (A)", "colorhist/colorhist_s0_oof.npz"),
             ("zeroshot (B)", "zeroshot/zeroshot_s0_oof.npz"), ("deepfake (C)", "deepfake/deepfake_s0_oof.npz"))}
have = _have(); print("[import] available:", have)

# --- B missing: run zero-shot HERE when this session already has the converted fire NPZs and a GPU
FIRE_NPZ = f"{os.environ.get('DATA', WK + '/npz')}/fire.npz"
if not have["zeroshot (B)"] and globals().get("AUTO_RUN_B", True):
    import torch
    if os.path.exists(FIRE_NPZ) and torch.cuda.is_available():
        import numpy as np
        n_rows = len(np.load(FIRE_NPZ, allow_pickle=True)["labels"])
        print(f"[auto-B] B (zero-shot) attached nahi hai → isi session me chala rahe hain on {FIRE_NPZ} (n={n_rows}); ~10–20 min")
        sh("pip -q install open_clip_torch", tag="pip open_clip")
        sh(f"set -o pipefail; bash {CODE}/trinetra/kaggle/run_hackathon.sh zeroshot 2>&1 | tee {WK}/zeroshot.log",
           timeout_min=TIMEOUT_MIN.get("zeroshot", 60), tag="zeroshot (auto-B)")
        have = _have(); print("[import] available after auto-B:", have)
    else:
        print(f"[auto-B] not possible here: fire.npz={'yes' if os.path.exists(FIRE_NPZ) else 'no'}, "
              f"GPU={'yes' if torch.cuda.is_available() else 'no'}")

def _attach_help(missing):
    print("\n" + "=" * 78)
    print(f"[!] {missing} ka result nahi mila. Kya karna hai:")
    print("    • Isi session me A chala tha to out_A_%s.zip /kaggle/working me hona chahiye (Output panel dekho)." % PHASE)
    print("    • Doosre account ka run: right panel → Add Input → Your Work (ya Notebooks) → wo run → Add → Run All.")
    print("    • Ya settings cell me EXTRA_RUN_DIRS = ['/kaggle/input/...'] daalo.")
    print("=" * 78 + "\n")

if not have["fire (A)"]:
    _attach_help("A (fire)")
    raise SystemExit("Fusion ke liye A ka fire OOF (fire/fire_s0_oof.npz) chahiye.")
if not have["zeroshot (B)"]:
    if globals().get("REQUIRE_B", True):
        _attach_help("B (zero-shot)")
        raise SystemExit("B (zero-shot) nahi mila aur yahan chal bhi nahi saka. B attach karo — ya sirf fire + colour baseline se "
                         "fusion chahiye to settings cell me REQUIRE_B = False karo.")
    if not have["colorhist (A)"]:
        raise SystemExit("B nahi aur A ka colour-histogram OOF bhi nahi: fusion ke liye doosra source chahiye.")
    print("[warn] B (zero-shot) nahi hai: REQUIRE_B=False, isliye fusion = fire + colour baseline (kamzor comparison).")
if not have["deepfake (C)"]:
    print("[note] C (deepfake) nahi: fusion media-authenticity source ke bina chalega (allowed, C preferred).")

In [ ]:
# 3) train / evaluate for this account; results are collected after every stage so a timeout keeps partial outputs
R = f"bash {CODE}/trinetra/kaggle/run_hackathon.sh"
plans = {"A": ["fire", "video"], "B": ["zeroshot"], "C": ["deepfake"], "D": ["eo-train", "geo"], "F": ["fuse"], "P": ["physics"],
         "ALL": ["fire", "deepfake", "zeroshot", "fuse", "video", "eo-train", "physics"]}
status = {}
for stage in plans[ACCOUNT]:
    need = NEEDS.get(stage)
    if need and not any(os.path.exists(f"{os.environ['DATA']}/{f}") for f in need):
        print(f"[skip] {stage}: none of {need} in {os.environ['DATA']} (see the data cell above)"); status[stage] = "skipped: no input"
        continue
    if stage == "fuse" and not os.path.exists("/kaggle/working/out/fire/fire_s0_oof.npz"):
        print("[skip] fuse: needs /kaggle/working/out/fire/fire_s0_oof.npz from the fire stage"); status[stage] = "skipped: no fire OOF"
        continue
    status[stage] = sh(f"set -o pipefail; {R} {stage} 2>&1 | tee /kaggle/working/{stage}.log",
                       timeout_min=TIMEOUT_MIN.get(stage), tag=stage)
    sh(f"python {CODE}/trinetra/kaggle/collect_results.py --inp /kaggle/working/out --out /kaggle/working/out/results > /dev/null 2>&1")
print("stage exit codes:", status, f"· total {(time.time() - T_RUN) / 60:.0f} min")
open("/kaggle/working/stage_status.json", "w").write(json.dumps(status, indent=2))

In [ ]:
# 4) collect + package (download out.zip or publish /kaggle/working/out as a private dataset version)
sh(f"python {CODE}/trinetra/kaggle/collect_results.py --inp /kaggle/working/out --out /kaggle/working/out/results")
sh("cp /kaggle/working/*.log /kaggle/working/stage_status.json /kaggle/working/out/ 2>/dev/null; cp -r /kaggle/working/trinetra_stage/*.json /kaggle/working/out/ 2>/dev/null")
shutil.make_archive(f"/kaggle/working/out_{ACCOUNT}_{PHASE}", "zip", "/kaggle/working/out")
sh("cat /kaggle/working/out/results/RESULTS.md 2>/dev/null | head -80")
if not any(v == 0 for v in status.values()):
    print("\n[!] No stage finished successfully, so RESULTS.md has no rows. Fix the first [warn]/[skip] above and Run All again.")
print("download:", f"/kaggle/working/out_{ACCOUNT}_{PHASE}.zip")

In [ ]:
# 5) MASTER RESULTS: merge every attached run (A, B, C, D, P + this fusion run) into ONE scorecard.
#    One phase per account (this notebook's PHASE first, else the other); repo results (landslide physics, GLOF, RL env) are added.
#    If an attached P/D run was made with older code (no landslide meta-fusion yet), the meta-fusion is computed HERE from that
#    run's own geo_events.csv + Geo-PINN OOF files, so P does not have to be re-run.
#    Output: /kaggle/working/master/MASTER_RESULTS.md (+ RESULTS.md, master.json), zipped as master_<PHASE>.zip. Send it to the AI.
import zipfile
# search ONLY the attached run outputs (run_roots from the import cell), never the 61 GB team dataset
zips = []
for r in run_roots:
    for cur, dirs, files in os.walk(r, followlinks=True):
        zips += [os.path.join(cur, f) for f in files if re.match(r"out_.*\.zip$", f)]
zips = sorted(set(zips + glob.glob("/kaggle/working/out_*.zip")))   # + runs made in THIS session (e.g. A) and this F run
print("[master] runs found:", [os.path.basename(z) for z in zips])

GEO = "/kaggle/working/geo_fusion_addon"; extra_dirs = []
def _phase_rank(z):
    return 2 if z.endswith(f"_{PHASE}.zip") else 1
for acc in ("P", "D", "ALL"):
    cand = sorted([z for z in zips if os.path.basename(z).startswith(f"out_{acc}_")], key=_phase_rank, reverse=True)
    if not cand:
        continue
    z = cand[0]; dst = f"{GEO}/src_{acc}"
    with zipfile.ZipFile(z) as zf:
        names = zf.namelist()
        if any("geo_fusion_s" in n for n in names):
            print(f"[geo-fusion] {os.path.basename(z)} already contains the meta-fusion"); break
        if not any(n.endswith("geo_events.csv") for n in names):
            continue
        zf.extractall(dst)
    ev = (glob.glob(f"{dst}/**/events/geo_events.csv", recursive=True) or glob.glob(f"{dst}/**/geo_events.csv", recursive=True))[0]
    pinns = sorted(glob.glob(f"{dst}/**/geo_pinn_s*_oof.npz", recursive=True))
    seeds = sorted({int(re.findall(r"_s(\d+)_oof", p)[0]) for p in pinns}) or [0]
    out_dir = f"{GEO}/out_{acc}/geo/fusion"
    for s in seeds:
        pin = next((p for p in pinns if f"_s{s}_oof" in p), "")
        sh(f"python {CODE}/trinetra/kaggle/geo_meta_fusion.py --events '{ev}' " + (f"--pinn '{pin}' " if pin else "") + f"--out '{out_dir}' --seed {s}",
           timeout_min=TIMEOUT_MIN["geo"], tag=f"geo-fusion s{s}")
    extra_dirs.append(f"{GEO}/out_{acc}")
    print(f"[geo-fusion] computed from {os.path.basename(z)} for seeds {seeds}")
    break

sh(f"python {CODE}/trinetra/kaggle/merge_all_results.py --prefer {PHASE} --zips " + " ".join(f"'{z}'" for z in zips)
   + (" --dirs " + " ".join(f"'{d}'" for d in extra_dirs) if extra_dirs else "") + " --out /kaggle/working/master", timeout_min=TIMEOUT_MIN["merge"], tag="merge")
shutil.make_archive(f"/kaggle/working/master_{PHASE}", "zip", "/kaggle/working/master")
print("download: /kaggle/working/master_%s.zip" % PHASE)